In [9]:
import pyspark.sql.functions as F

df = spark.read.table("dbo.taxi_rides_copy_activity_zorder")

min_dt, max_dt = df.agg(
    F.min("lpepPickupDatetime"),
    F.max("lpepPickupDatetime")
).first()

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 11, Finished, Available, Finished, False)

In [ ]:
print(min_dt)
print(max_dt)

In [10]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

start = min_dt.replace(hour=0, minute=0, second=0, microsecond=0)
end = max_dt.replace(hour=0, minute=0, second=0, microsecond=0)

date_list = [(start + timedelta(days=i),) for i in range((end - start).days + 1)]
df = spark.createDataFrame(date_list, ["Datetime"])
df = df.withColumn("Date", col("Datetime").cast(DateType())) \
       .withColumn("Year", year(col("Datetime"))) \
       .withColumn("Month", month(col("Datetime"))) \
       .withColumn("Quarter", quarter(col("Datetime"))) \
       .withColumn("MonthName", date_format(col("Datetime"), "MMMM")) \
       .withColumn("DayName", date_format(col("Datetime"), "EEEE"))

dfh = (
       spark.read.table("external.public_holidays")
       .filter(col("countryRegionCode") == "US")
       .select(col("date"), col("holidayName"))
       .withColumn("date", to_date(col("date")))
       .withColumnRenamed("date", "HolidayDate")
       .withColumnRenamed("holidayName", "holiday")
)

df = df.join(dfh, df.Date == dfh.HolidayDate, "left").drop("HolidayDate")

# display(df)

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 12, Finished, Available, Finished, False)

In [ ]:
print(df.count())
df.printSchema()
display(df)

In [11]:
df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 13, Finished, Available, Finished, False)

In [1]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimLocation.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_location_from")

StatementMeta(, 6b37809f-00e2-46a9-9c0e-f8c8db2760bf, 3, Finished, Available, Finished, False)

In [2]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimLocation.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_location_to")

StatementMeta(, 6b37809f-00e2-46a9-9c0e-f8c8db2760bf, 4, Finished, Available, Finished, False)

In [13]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimRateCode.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_ratecode")

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 15, Finished, Available, Finished, False)

In [14]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimTripType.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_triptype")

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 16, Finished, Available, Finished, False)

In [15]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimVendor.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_vendor")

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 17, Finished, Available, Finished, False)

In [16]:
df = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/DimPaymentType.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_paymenttype")

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 18, Finished, Available, Finished, False)

In [2]:
%%sql
CREATE OR REPLACE TABLE gold.fact_trips
USING DELTA
AS
SELECT vendorID
      ,lpepPickupDatetime AS PickupDatetime
      ,lpepDropoffDatetime AS DropoffDatetime
      ,CAST(lpepPickupDatetime AS date) AS PickupDate
      ,CAST(lpepDropoffDatetime AS date) AS DropoffDate
      ,DATEDIFF(SECOND, lpepPickupDatetime,lpepDropoffDatetime) AS Duration
      ,passengerCount
      ,tripDistance
      ,CAST(puLocationId AS int) AS puLocationId
      ,CAST(doLocationId AS int) AS doLocationId
      ,pickupLongitude
      ,pickupLatitude
      ,dropoffLongitude
      ,dropoffLatitude
      ,rateCodeID
      ,storeAndFwdFlag
      ,paymentType
      ,fareAmount
      ,extra
      ,mtaTax
      ,improvementSurcharge
      ,tipAmount
      ,tollsAmount
      ,ehailFee
      ,totalAmount
      ,tripType
  FROM dbo.taxi_rides_copy_activity_zorder

StatementMeta(, 6b22d7e1-dc19-4a5b-ab65-151173f1c734, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
print(spark.sql("DESCRIBE DETAIL gold.fact_trips").collect()[0])

StatementMeta(, a70156e0-0956-4cda-90b4-f42bc0b3fbea, 20, Finished, Available, Finished, False)

Row(format='delta', id='43678e71-b8e9-42a9-b432-dce4ffa6db4d', name='spark_catalog.chimcobldhq2ajip8cg58obod4iksma3ahgngqac90imerrccg.fact_trips', description=None, location='abfss://8cdf1706-1fd1-4985-9125-a22e233cd933@onelake.dfs.fabric.microsoft.com/5838af0b-f450-4a52-8644-a7269ea77912/Tables/gold/fact_trips', createdAt=datetime.datetime(2026, 8, 7, 19, 44, 19, 856000), lastModified=datetime.datetime(2026, 8, 7, 19, 45, 15, 651000), partitionColumns=[], clusteringColumns=[], numFiles=13, sizeInBytes=1738004020, properties={'delta.stats.extended.collect': 'true', 'delta.stats.extended.inject': 'true'}, minReaderVersion=1, minWriterVersion=2, tableFeatures=['appendOnly', 'invariants'])


In [4]:
import builtins

tables_df = spark.sql("SHOW TABLES IN gold")
tables = [row.tableName for row in tables_df.collect()]

results = []
for tbl in tables:
    tbl_name = f"gold.{tbl}"
    details = spark.sql(f"DESCRIBE DETAIL {tbl_name}").collect()[0]
    results.append({
        "table": tbl,
        "size_gb": builtins.round(details["sizeInBytes"] / (1024**3), 2),
        "num_files": details["numFiles"],
        "avg_file_size_mb": builtins.round((details["sizeInBytes"] / details["numFiles"]) / (1024**2), 2)
    })

summary_df = spark.createDataFrame(results).select(
    "table", "size_gb", "num_files", "avg_file_size_mb"
)
display(summary_df)

StatementMeta(, 6b37809f-00e2-46a9-9c0e-f8c8db2760bf, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc97b339-c1ac-4b67-9e61-ef8b374ed85f)

In [3]:
spark.sql("OPTIMIZE gold.fact_trips ZORDER BY (PickupDate) VORDER;")

StatementMeta(, 6b22d7e1-dc19-4a5b-ab65-151173f1c734, 4, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,

In [15]:
tables_df = spark.sql("SHOW TABLES IN gold")
dim_tables = [row.tableName for row in tables_df.collect() if row.tableName.startswith("dim")]

print(f"Found {len(dim_tables)} dim tables: {dim_tables}")

for tbl in dim_tables:
    full_name = f"gold.{tbl}"
    details = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0]
    if details["numFiles"] > 1:
        print(f"Optimizing {full_name} ({details['numFiles']} files)")
        spark.sql(f"OPTIMIZE {full_name}")
    else:
        print(f"Skipping {full_name} — already 1 file, no benefit")

StatementMeta(, 4a01485c-1a4b-4474-9772-885cfefc61a3, 17, Finished, Available, Finished, False)

Found 6 dim tables: ['dim_date', 'dim_location', 'dim_paymenttype', 'dim_ratecode', 'dim_triptype', 'dim_vendor']
Optimizing gold.dim_date (8 files)
Skipping gold.dim_location — already 1 file, no benefit
Skipping gold.dim_paymenttype — already 1 file, no benefit
Skipping gold.dim_ratecode — already 1 file, no benefit
Skipping gold.dim_triptype — already 1 file, no benefit
Skipping gold.dim_vendor — already 1 file, no benefit


In [3]:
tables_df = spark.sql("SHOW TABLES IN gold")
dim_tables = [row.tableName for row in tables_df.collect() if row.tableName.startswith("dim")]

print(f"Found {len(dim_tables)} dim tables: {dim_tables}")

for tbl in dim_tables:
    full_name = f"gold.{tbl}"
    details = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0]
    print(f"Optimizing {full_name} ({details['numFiles']} files)")
    spark.sql(f"OPTIMIZE {full_name} VORDER;")

StatementMeta(, 6b37809f-00e2-46a9-9c0e-f8c8db2760bf, 5, Finished, Available, Finished, False)

Found 7 dim tables: ['dim_date', 'dim_location_from', 'dim_location_to', 'dim_paymenttype', 'dim_ratecode', 'dim_triptype', 'dim_vendor']
Optimizing gold.dim_date (1 files)
Optimizing gold.dim_location_from (1 files)
Optimizing gold.dim_location_to (1 files)
Optimizing gold.dim_paymenttype (1 files)
Optimizing gold.dim_ratecode (1 files)
Optimizing gold.dim_triptype (1 files)
Optimizing gold.dim_vendor (1 files)
